### tools

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool

model=init_chat_model("ollama:qwen3:8b")

# @tool turns a function into a tool. The type hints define its input schema and the
# docstring becomes the description the LLM reads to decide when to call it (NOTES.md §8, §10).
@tool
def get_weather(location:str)->str:
    """Get weather info of a location"""
    return f"It is sunny in {location}."

# bind_tools gives the model native tool-calling: it can now emit tool_calls on an
# AIMessage instead of answering directly. Binding does NOT run anything yet —
# the next cell shows the manual loop that actually executes the call.
model_with_tools=model.bind_tools([get_weather])


### tool execution loop

In [ ]:
# The tool-execution loop, done by hand (what create_agent / a StateGraph automates):
#   HumanMessage -> AIMessage(tool_calls) -> ToolMessage(result) -> AIMessage(answer)
# This is the agent message cycle from NOTES.md §8.
messages = [{"role":"user","content":"what is weather in london?"}]
ai_msg = model_with_tools.invoke(messages)   # model decides to call get_weather
messages.append(ai_msg)

# Run each requested tool and feed its result back as a ToolMessage. Passing the whole
# tool_call to .invoke() lets LangChain attach the matching tool_call_id automatically.
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Second model call: now that it can see the tool result, it writes the final answer.
final_response = model_with_tools.invoke(messages)
final_response.text